# 05 — Domain Classifier (Pre-filter)

**Project:** TalentMatch.ai — Iteration 2  
**Goal:** Train a binary domain classifier that detects whether a resume and job description are from the same professional domain. This acts as a pre-filter in the demo: if domains don't match, we skip the regression model and return a low score immediately.

## Why this helps
The regression model struggles most with cross-domain pairs (nurse → backend engineer). The domain classifier handles this case explicitly — it's a much simpler binary decision that DistilBERT should solve with >90% accuracy given 37,740 training samples.

## Two-model pipeline
```
User input (resume + JD)
    → Domain Classifier
        → Different domain: return score ~10-20 (low fit)
        → Same domain: pass to regression model → return 0-100 score
```

## Dataset
- `0xnbk/resume-domain-classifier-v1-en`
- Only has `validation` split — we treat it as the full dataset and do our own 80/20 split
- 37,740 samples, binary label (1 = same domain, 0 = different domain)
- Derived from real LinkedIn job postings

## Notebook Flow
1. Install dependencies + authenticate
2. Load dataset and inspect schema
3. Parse and split into train/val
4. Build PyTorch Dataset
5. Load DistilBERT with binary classification head
6. Train with Trainer API
7. Evaluate: accuracy, precision, recall, F1
8. Push classifier to HuggingFace Hub
9. Test the full two-model pipeline


In [ ]:
# =============================================================
# Step 1 — Install dependencies and authenticate
# =============================================================
# REMINDER: HF_TOKEN must be set in Colab Secrets with write access.
# Runtime must be T4 GPU.
# =============================================================

!pip install -q -U transformers accelerate huggingface_hub evaluate scikit-learn

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Authenticated with HuggingFace Hub.")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================
# Step 2 — Load dataset and inspect schema
# =============================================================
# This dataset only has a 'validation' split (confirmed from
# exploration). We load it and treat it as our full dataset.
#
# Schema:
#   text:         resume + ' SEP ' + jd (same format as dataset 1)
#   label:        1 = same domain, 0 = different domain
#   pair_type:    'match' or 'mismatch'
#   resume_domain: string (e.g. 'Technology', 'Healthcare')
#   job_domain:   string
# =============================================================

from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("0xnbk/resume-domain-classifier-v1-en")
print(dataset)

# The only split is 'validation' — treat it as full dataset
sample = dataset['validation'][0]
print(f"\nFields: {list(sample.keys())}")
print(f"Label: {sample['label']}")
print(f"Pair type: {sample['pair_type']}")
print(f"Resume domain: {sample['resume_domain']}")
print(f"Job domain: {sample['job_domain']}")
print(f"Text (first 200 chars): {sample['text'][:200]}")

In [ ]:
# =============================================================
# Step 3 — Parse text and split into train/val
# =============================================================
# Same parsing as notebook 04: split on ' SEP '.
#
# We do an 80/20 stratified split to ensure both train and val
# have balanced label distributions (equal match/mismatch ratio).
# Stratification is important here because label imbalance in
# either split would give misleading accuracy metrics.
# =============================================================

from sklearn.model_selection import train_test_split

def parse_sample(sample):
    """Split text into resume and JD. Returns None if malformed."""
    text = sample['text']
    if ' SEP ' not in text:
        return None
    parts = text.split(' SEP ', 1)
    resume = parts[0].strip()
    jd     = parts[1].strip()
    if not resume or not jd:
        return None
    return {
        'resume':        resume,
        'jd':            jd,
        'label':         int(sample['label']),
        'resume_domain': sample['resume_domain'],
        'job_domain':    sample['job_domain'],
    }

records, skipped = [], 0
for s in dataset['validation']:
    parsed = parse_sample(s)
    if parsed is None:
        skipped += 1
    else:
        records.append(parsed)

df = pd.DataFrame(records)
print(f"Valid samples: {len(df)} | Skipped: {skipped}")
print(f"Label balance: {df['label'].value_counts().to_dict()}")

# Stratified 80/20 split
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")
print(f"Train label balance: {train_df['label'].value_counts().to_dict()}")
print(f"Val label balance:   {val_df['label'].value_counts().to_dict()}")

In [ ]:
# =============================================================
# Step 4 — Build PyTorch Dataset
# =============================================================
# For binary classification, labels are integers (0 or 1).
# No normalization needed — CrossEntropyLoss handles integers directly.
# =============================================================

from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DomainClassifierDataset(Dataset):
    """PyTorch Dataset for domain match classification."""
    def __init__(self, df, tokenizer):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['resume'],
            row['jd'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            # Integer label for CrossEntropyLoss
            'labels': torch.tensor(row['label'], dtype=torch.long)
        }

train_dataset = DomainClassifierDataset(train_df, tokenizer)
val_dataset   = DomainClassifierDataset(val_df,   tokenizer)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")

sample = train_dataset[0]
print(f"\ninput_ids shape: {sample['input_ids'].shape}")
print(f"label:           {sample['labels'].item()} (0=diff domain, 1=same domain)")

In [ ]:
# =============================================================
# Step 5 — Load DistilBERT with binary classification head
# =============================================================
# num_labels=2 → CrossEntropyLoss binary classification.
# Output: logits for [different_domain, same_domain]
# We take argmax to get the predicted class.
# =============================================================

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # binary classification: 0=diff domain, 1=same domain
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# =============================================================
# Step 6 — Define metrics and train
# =============================================================
# For binary classification we track:
#   Accuracy:  % of predictions correct overall
#   Precision: of predicted same-domain, how many actually are?
#   Recall:    of actual same-domain pairs, how many did we catch?
#   F1:        harmonic mean of precision and recall
#
# F1 is the most important metric here — we want to minimize
# both false positives (sending mismatched pairs to the scorer)
# and false negatives (blocking matched pairs from scoring).
#
# Expected: accuracy > 90%, F1 > 0.90
# =============================================================

from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive
drive.mount('/content/drive')

def compute_metrics(eval_pred):
    """Accuracy, precision, recall, F1 for binary domain classification."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    return {
        'accuracy':  round(float(accuracy),  4),
        'precision': round(float(precision), 4),
        'recall':    round(float(recall),    4),
        'f1':        round(float(f1),        4),
    }

OUTPUT_DIR = "/content/drive/MyDrive/TalentMatch-AI/checkpoints/domain_classifier"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,          # classification converges faster than regression
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting domain classifier training...")
print(f"Training samples: {len(train_dataset)}")
print(f"Steps per epoch:  {len(train_dataset) // training_args.per_device_train_batch_size}")
print("---")

train_result = trainer.train()

print("\nTraining complete!")
print(f"Training time: {train_result.metrics['train_runtime']:.1f}s")

In [ ]:
# =============================================================
# Step 7 — Evaluate
# =============================================================

print("Evaluating domain classifier on validation set...")
val_results = trainer.evaluate(eval_dataset=val_dataset)

print("\n===== DOMAIN CLASSIFIER RESULTS =====")
print(f"Accuracy:  {val_results['eval_accuracy']:.4f}")
print(f"Precision: {val_results['eval_precision']:.4f}")
print(f"Recall:    {val_results['eval_recall']:.4f}")
print(f"F1:        {val_results['eval_f1']:.4f}")
print("======================================")

if val_results['eval_accuracy'] >= 0.90:
    print("\n✓ Accuracy >= 90% — classifier is ready for production use.")
else:
    print("\n~ Accuracy < 90% — consider more epochs or checking data quality.")

In [ ]:
# =============================================================
# Step 8 — Push domain classifier to HuggingFace Hub
# =============================================================

CLASSIFIER_REPO = "LucasLisboadev/TalentMatch-DomainClassifier"

print(f"Pushing classifier to: https://huggingface.co/{CLASSIFIER_REPO}")

model.save_pretrained("/content/domain_classifier")
tokenizer.save_pretrained("/content/domain_classifier")

model.push_to_hub(CLASSIFIER_REPO)
tokenizer.push_to_hub(CLASSIFIER_REPO)

print(f"\nClassifier pushed!")
print(f"View at: https://huggingface.co/{CLASSIFIER_REPO}")

In [ ]:
# =============================================================
# Step 9 — Test the full two-model pipeline
# =============================================================
# This is the pipeline that will run in the updated Gradio demo:
#
#   1. Run domain classifier on resume + JD
#   2. If different domain → return low score (10-20 range)
#   3. If same domain → run regression model → return 0-100 score
#
# We test with the same 3 pairs as notebook 04.
# =============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np

# Load both models from Hub
reg_tokenizer  = AutoTokenizer.from_pretrained("LucasLisboadev/TalentMatch-AI-v2")
reg_model      = AutoModelForSequenceClassification.from_pretrained("LucasLisboadev/TalentMatch-AI-v2")
reg_model.eval()

cls_tokenizer  = AutoTokenizer.from_pretrained(CLASSIFIER_REPO)
cls_model      = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_REPO)
cls_model.eval()


def is_same_domain(resume, jd, tokenizer, model, threshold=0.5):
    """Returns True if resume and JD are in the same domain."""
    inputs = tokenizer(
        resume, jd,
        max_length=512, truncation=True, padding='max_length', return_tensors='pt'
    )
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    same_domain_prob = probs[0][1].item()  # probability of label=1 (same domain)
    return same_domain_prob >= threshold, round(same_domain_prob * 100, 1)


def predict_score_regression(resume, jd, tokenizer, model):
    """Run regression model. Returns score 0-100."""
    inputs = tokenizer(
        resume, jd,
        max_length=512, truncation=True, padding='max_length', return_tensors='pt'
    )
    with torch.no_grad():
        output = model(**inputs)
    score = output.logits.squeeze().item() * 100
    return round(float(np.clip(score, 0, 100)), 2)


def full_pipeline(resume, jd):
    """Full two-model scoring pipeline."""
    same_domain, domain_confidence = is_same_domain(
        resume, jd, cls_tokenizer, cls_model
    )
    if not same_domain:
        # Different domain — return low score without running regression
        score = 10.0
        return score, f"Cross-domain mismatch detected ({domain_confidence}% same-domain confidence) → Score capped at {score}"
    else:
        # Same domain — run full regression model
        score = predict_score_regression(resume, jd, reg_tokenizer, reg_model)
        return score, f"Same domain ({domain_confidence}% confidence) → Regression score: {score}/100"


# Test the three pairs
strong_resume = """Senior Backend Engineer with 6 years of experience building scalable
REST APIs using Python and FastAPI. Proficient in PostgreSQL, Redis, and Docker.
Led a team of 4 engineers to deliver a microservices architecture serving 2M daily users."""
strong_jd = """Backend Engineer: 4+ years Python experience required.
FastAPI or Django, PostgreSQL, Docker. Team leadership a plus."""

moderate_resume = """Marketing Manager with 5 years in B2B demand generation,
content strategy, Salesforce and HubSpot. Led campaigns generating $2M pipeline."""
moderate_jd = """Sales Director: 5+ years in B2B sales or marketing.
Salesforce CRM, pipeline management, executive presentations required."""

weak_resume = """Registered Nurse with 8 years ICU experience. Patient care,
medication administration, Epic EHR certified. ACLS/BLS certified."""
weak_jd = """Backend Engineer: 4+ years Python experience required.
FastAPI or Django, PostgreSQL, Docker."""

print("===== FULL PIPELINE TEST =====")
for name, resume, jd in [
    ("Strong match",   strong_resume,   strong_jd),
    ("Moderate match", moderate_resume, moderate_jd),
    ("Weak match",     weak_resume,     weak_jd),
]:
    score, explanation = full_pipeline(resume, jd)
    print(f"\n{name}:")
    print(f"  Score: {score}/100")
    print(f"  {explanation}")

print("\n============================")

## Summary

| Model | Task | HF Hub |
|---|---|---|
| TalentMatch-AI-v2 | Regression (0-100 score) | LucasLisboadev/TalentMatch-AI-v2 |
| TalentMatch-DomainClassifier | Binary domain match | LucasLisboadev/TalentMatch-DomainClassifier |

**Next step:** Update `app.py` in the HuggingFace Space to use both models via the two-model pipeline.
